# Modelos y Métricas Básicas
Buscamos eventualmente poder responder: 
> ¿De qué manera la transición al modelo Tec21 alteró la heterogeneidad estructural de la deserción entre escuelas, y qué variables han ganado peso predictivo en este nuevo régimen?

Se evalúan tres modelos:

    M1 (Baseline): Regresión logística penalizada estándar.

    M2 (Régimen Explícito): Regresión logística con interacciones clave era × variable para detectar cambios de peso predictivo.

    M3 (Multinivel): Modelo jerárquico Bayesiano con interceptos aleatorios por escuela y pendientes aleatorias de la era por escuela.

Los datos se dividen estrictamente en Entrenamiento (60%), Validación (20%) y Prueba (20%).

In [ ]:
import json
import warnings
from pathlib import Path
from time import perf_counter

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import joblib

from scipy.special import expit
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay, f1_score
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, train_test_split, learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
DATA_DIR = Path("../data/processed")
ARTIFACTS = Path("../outputs/model_artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

## Carga de Datos y Split Estricto
Cargamos el dataset y el manifiesto. Realizamos una partición estratificada considerando tanto la era como el target, y extraemos los grupos de escuela para evitar data leakage durante la validación cruzada.

In [ ]:
df = pd.read_parquet(DATA_DIR / "dataset_estudio_desercion.parquet")

with open(DATA_DIR / "feature_manifest.json", encoding="utf-8") as f:
    M = json.load(f)

TARGET = M["target"]
NUM_COMMON = M["numeric_common"]
CAT_COMMON = M["categorical_common"]
ERA_COL = "era_code"
SCHOOL_COL = "school_code"

# Estratificación combinada para mantener la proporción de deserciones por era
df["_strat"] = df[ERA_COL].astype(str) + "_" + df[TARGET].astype(int).astype(str)

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int).values

# 1. Separar el Test Set (20%) - Congelado hasta el final
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=df["_strat"]
)

# 2. Separar Train (60%) y Validation (20%)
strat_dev = (X_dev[ERA_COL].astype(str).values + "_" + y_dev.astype(str))
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=SEED, stratify=strat_dev
)


def assert_school_coverage(train_df, other_df, school_col, split_name):
    unseen = sorted(set(other_df[school_col]) - set(train_df[school_col]))
    if unseen:
        raise ValueError(
            f"{split_name} contiene {len(unseen)} escuelas no vistas en train. "
            f"Ejemplos: {unseen[:10]}"
        )


assert_school_coverage(X_train, X_val, SCHOOL_COL, "Validation")
assert_school_coverage(X_train, X_test, SCHOOL_COL, "Test")

## Cargar helpers

In [ ]:
# Preprocesamiento y Funciones de Evaluación

def build_preprocessor(numeric_cols, categorical_cols):
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", sparse_output=False, min_frequency=30), categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def find_threshold_at_recall(y_true, y_prob, target_recall=0.75):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    best_t = thresholds[0]
    for r, t in zip(recall[:-1], thresholds):
        if r >= target_recall:
            best_t = t
    return float(best_t)

def find_optimal_threshold_f1(y_true, y_prob):
    """
    Encuentra el umbral probabilístico que maximiza el F1-Score 
    para conjuntos de datos desbalanceados.
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    
    # Evitar divisiones por cero con np.divide
    f1_scores = np.divide(
        2 * (precision * recall), 
        (precision + recall), 
        out=np.zeros_like(precision), 
        where=(precision + recall) != 0
    )
    
    # Obtener el índice del F1 máximo
    best_idx = np.argmax(f1_scores)
    
    # precision y recall tienen un elemento más que thresholds, 
    # por lo que aseguramos no salirnos del índice.
    if best_idx == len(thresholds):
        best_idx -= 1
        
    return float(thresholds[best_idx])

def bootstrap_ci(metric_fn, y_true, y_prob, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = [metric_fn(y_true[idx], y_prob[idx]) for _ in range(n_boot) if (idx := rng.integers(0, len(y_true), len(y_true))) is not None and y_true[idx].sum() > 0]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def evaluate_probs(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "roc_auc_ci": bootstrap_ci(roc_auc_score, y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "pr_auc_ci": bootstrap_ci(average_precision_score, y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "threshold": threshold,
        "cm": confusion_matrix(y_true, y_pred),
    }

def search_results_table(search, top_n=5):
    cols = [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_fit_time",
        "params",
    ]
    return (
        pd.DataFrame(search.cv_results_)[cols]
        .sort_values(["rank_test_score", "mean_test_score"])
        .head(top_n)
        .reset_index(drop=True)
    )

# M1 - Regresión Logística Base

Establece la línea base predictiva global (sin efectos estructurales explícitos por escuela ni era). Usamos validación cruzada agrupada por escuela optimizando PR-AUC

In [ ]:
preprocessor_m1 = build_preprocessor(NUM_COMMON + [ERA_COL], CAT_COMMON)

m1_pipe = Pipeline([
    ("prep", preprocessor_m1),
    ("clf", LogisticRegression(solver="saga", penalty="elasticnet", max_iter=4000, random_state=SEED))
])

m1_space = {
    "prep__cat__min_frequency": [10, 30, 50],
    "clf__C": loguniform(1e-3, 1e2),
    "clf__l1_ratio": np.linspace(0.0, 1.0, 11),
    "clf__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
m1_search = RandomizedSearchCV(
    m1_pipe, m1_space, n_iter=30, scoring="average_precision", cv=cv, random_state=SEED, n_jobs=-1, verbose=1
)

t0 = perf_counter()
m1_search.fit(X_train, y_train)
m1_train_time = perf_counter() - t0
m1_best = m1_search.best_estimator_

# Evaluación en Validación
prob_val_m1 = m1_best.predict_proba(X_val)[:, 1]
thr_m1 = find_optimal_threshold_f1(y_val, prob_val_m1)
m1_metrics = evaluate_probs(y_val, prob_val_m1, thr_m1)

### Resumen CV

In [ ]:
m1_cv_top = search_results_table(m1_search, top_n=5)
m1_cv_top.to_csv(ARTIFACTS / "m1_cv_top_results.csv", index=False)

print("M1 best params:")
print(m1_search.best_params_)
print(f"M1 training time: {m1_train_time:.2f} s")
display(m1_cv_top)

# M2 — Regresión Logística con Interacciones Tec21

Identifica variables cuyo peso cambió en el nuevo modelo educativo. Calculamos la interacción matemáticamente como el producto cruzado de la variable y la bandera binaria de Tec21.

In [ ]:
# Variables universales con sentido teórico para interactuar con la época
M2_INTERACTIONS = ["PNA", "scholarship.perc", "activity_count_unified", "admission.test"]

def add_m2_features(df_in):
    out = df_in.copy()
    out["era_main"] = out[ERA_COL]
    for col in M2_INTERACTIONS:
        if col in out.columns:
            out[f"{col}_x_era"] = out[col].fillna(0) * out[ERA_COL]
    return out

X_train_m2 = add_m2_features(X_train)
X_val_m2 = add_m2_features(X_val)

num_m2 = NUM_COMMON + ["era_main"] + [f"{c}_x_era" for c in M2_INTERACTIONS]
preprocessor_m2 = build_preprocessor(num_m2, CAT_COMMON)

m2_pipe = Pipeline([
    ("prep", preprocessor_m2),
    ("clf", LogisticRegression(solver="saga", penalty="elasticnet", max_iter=4000, random_state=SEED))
])

m2_search = RandomizedSearchCV(
    m2_pipe, m1_space, n_iter=30, scoring="average_precision", cv=cv, random_state=SEED, n_jobs=-1, verbose=1
)

t0 = perf_counter()
m2_search.fit(X_train_m2, y_train)
m2_train_time = perf_counter() - t0

m2_best = m2_search.best_estimator_

# Evaluación en Validación
prob_val_m2 = m2_best.predict_proba(X_val_m2)[:, 1]
thr_m2 = find_optimal_threshold_f1(y_val, prob_val_m2)
m2_metrics = evaluate_probs(y_val, prob_val_m2, thr_m2)

### Resumen CV

In [ ]:
m2_cv_top = search_results_table(m2_search, top_n=5)
m2_cv_top.to_csv(ARTIFACTS / "m2_cv_top_results.csv", index=False)

print("M2 best params:")
print(m2_search.best_params_)
print(f"M2 training time: {m2_train_time:.2f} s")
display(m2_cv_top)

# M3 — Modelo Multinivel Jerárquico
Implementación en PyMC. Modela explícitamente el intercepto aleatorio $\alpha_j$ (riesgo base por escuela) y la pendiente aleatoria $\delta_j$ (efecto diferencial de la transición a Tec21 por escuela).

In [ ]:
# Preparar matrices para PyMC
Xtr_m3 = preprocessor_m2.fit_transform(X_train_m2).astype("float32")
Xva_m3 = preprocessor_m2.transform(X_val_m2).astype("float32")
feature_names_m3 = preprocessor_m2.get_feature_names_out()

# Índices para efectos aleatorios
school_index = {k: i for i, k in enumerate(sorted(df[SCHOOL_COL].unique()))}
school_train = X_train[SCHOOL_COL].map(school_index).values.astype("int32")
school_val = X_val[SCHOOL_COL].map(school_index).values.astype("int32")

era_train = X_train[ERA_COL].values.astype("int32")
era_val = X_val[ERA_COL].values.astype("int32")
n_schools = len(school_index)

coords = {
    "obs": np.arange(len(y_train)),
    "feature": feature_names_m3,
    "school": np.arange(n_schools),
}

t0 = perf_counter()

with pm.Model(coords=coords) as m3_model:
    X_data = pm.Data("X", Xtr_m3, dims=("obs", "feature"))
    school_idx = pm.Data("school_idx", school_train, dims="obs")
    era_idx = pm.Data("era_idx", era_train, dims="obs")

    # Intercepto aleatorio (Riesgo base por escuela)
    mu_alpha = pm.Normal("mu_alpha", 0.0, 1.5)
    sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)
    z_alpha = pm.Normal("z_alpha", 0.0, 1.0, dims="school")
    alpha = pm.Deterministic("alpha", mu_alpha + sigma_alpha * z_alpha, dims="school")

    # Pendiente aleatoria (Impacto de Tec21 por escuela)
    mu_delta = pm.Normal("mu_delta", 0.0, 1.0)
    sigma_delta = pm.HalfNormal("sigma_delta", 1.0)
    z_delta = pm.Normal("z_delta", 0.0, 1.0, dims="school")
    delta = pm.Deterministic("delta", mu_delta + sigma_delta * z_delta, dims="school")

    # Efectos fijos (Covariables comunes)
    beta = pm.Normal("beta", 0.0, 1.0, dims="feature")

    # Ecuación Logit
    logit_p = alpha[school_idx] + delta[school_idx] * era_idx + pm.math.dot(X_data, beta)
    
    # Likelihood
    pm.Bernoulli("y_obs", logit_p=logit_p, observed=y_train)

    # Inferencia rápida para desarrollo (Reemplazar por pm.sample para resultados finales)
    approx = pm.fit(n=30000, method="advi", random_seed=SEED, progressbar=True)
    trace_m3 = approx.sample(1000, random_seed=SEED)

m3_train_time = perf_counter() - t0

# Proyección en Validación
alpha_mean = trace_m3.posterior["alpha"].mean(dim=["chain", "draw"]).values
delta_mean = trace_m3.posterior["delta"].mean(dim=["chain", "draw"]).values
beta_mean = trace_m3.posterior["beta"].mean(dim=["chain", "draw"]).values

prob_val_m3 = expit(alpha_mean[school_val] + delta_mean[school_val] * era_val + Xva_m3 @ beta_mean)
thr_m3 = find_optimal_threshold_f1(y_val, prob_val_m3)
m3_metrics = evaluate_probs(y_val, prob_val_m3, thr_m3)

print(f"M3 training time: {m3_train_time:.2f} s")

# Comparación y Selección

Comparamos visualmente el desempeño en el conjunto de validación.

### Helpers

In [ ]:
def plot_learning_curve(
    estimator,
    X,
    y,
    cv,
    title,
    scoring="average_precision",
    save_path=None,
):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator=estimator,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        train_sizes=np.linspace(0.1, 1.0, 5),
        n_jobs=-1,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    plt.figure(figsize=(7, 4.5))
    plt.plot(train_sizes, train_mean, marker="o", label="Train")
    plt.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.15,
    )

    plt.plot(train_sizes, val_mean, marker="o", label="Validation")
    plt.fill_between(
        train_sizes,
        val_mean - val_std,
        val_mean + val_std,
        alpha=0.15,
    )

    plt.title(title, fontsize=12, fontweight="bold")
    plt.xlabel("Tamaño de entrenamiento")
    plt.ylabel("PR-AUC")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300)

    plt.show()


def error_audit(
    X_df,
    y_true,
    y_prob,
    threshold,
    school_col,
    era_col,
    numeric_cols=None,
):
    audit = X_df[[school_col, era_col]].reset_index(drop=True).copy()
    audit["y_true"] = y_true
    audit["y_prob"] = y_prob
    audit["y_pred"] = (y_prob >= threshold).astype(int)

    if numeric_cols is not None:
        keep_cols = [c for c in numeric_cols if c in X_df.columns]
        for col in keep_cols:
            audit[col] = X_df[col].reset_index(drop=True)

    conditions = [
        (audit["y_true"] == 1) & (audit["y_pred"] == 1),
        (audit["y_true"] == 0) & (audit["y_pred"] == 0),
        (audit["y_true"] == 0) & (audit["y_pred"] == 1),
        (audit["y_true"] == 1) & (audit["y_pred"] == 0),
    ]
    labels = ["TP", "TN", "FP", "FN"]
    audit["error_type"] = np.select(conditions, labels, default="UNK")

    by_era = (
        audit.groupby([era_col, "error_type"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    by_school = (
        audit.groupby(school_col)
        .apply(
            lambda g: pd.Series(
                {
                    "n": len(g),
                    "positives": int(g["y_true"].sum()),
                    "FP": int(
                        ((g["y_true"] == 0) & (g["y_pred"] == 1)).sum()
                    ),
                    "FN": int(
                        ((g["y_true"] == 1) & (g["y_pred"] == 0)).sum()
                    ),
                    "avg_prob": float(g["y_prob"].mean()),
                }
            )
        )
        .reset_index()
    )

    by_school["fn_rate_among_positives"] = np.divide(
        by_school["FN"],
        by_school["positives"],
        out=np.zeros(len(by_school), dtype=float),
        where=by_school["positives"] > 0,
    )

    by_school["fp_rate_overall"] = by_school["FP"] / by_school["n"]

    return audit, by_era, by_school.sort_values(
        "fn_rate_among_positives",
        ascending=False,
    )

### Resultados básicos

In [ ]:
results = pd.DataFrame([
    {"Modelo": "M1 (Baseline)", "Tiempo Entrenando": int(m1_train_time), "PR-AUC": m1_metrics["pr_auc"], "ROC-AUC": m1_metrics["roc_auc"], "Brier": m1_metrics["brier"]},
    {"Modelo": "M2 (Interacciones Tec21)", "Tiempo Entrenando": int(m2_train_time), "PR-AUC": m2_metrics["pr_auc"], "ROC-AUC": m2_metrics["roc_auc"], "Brier": m2_metrics["brier"]},
    {"Modelo": "M3 (Multinivel Hierárquico)", "Tiempo Entrenando": int(m3_train_time), "PR-AUC": m3_metrics["pr_auc"], "ROC-AUC": m3_metrics["roc_auc"], "Brier": m3_metrics["brier"]},
])

display(results.sort_values("PR-AUC", ascending=False))

## Tabla de comparación

In [ ]:
comparison_table = pd.DataFrame(
    [
        {
            "Model": "M1 (Baseline)",
            "AUC-ROC": m1_metrics["roc_auc"],
            "PR-AUC": m1_metrics["pr_auc"],
            "Precision": m1_metrics["precision"],
            "Recall": m1_metrics["recall"],
            "F1": m1_metrics["f1"],
            "Interpretabilidad": "Alta",
            "Tiempo entrenamiento (s)": m1_train_time,
            "Ventajas": (
                "Simple, rápido, coeficientes interpretables"
            ),
            "Limitaciones": (
                "No modela explícitamente cambio de régimen ni "
                "heterogeneidad entre escuelas"
            ),
        },
        {
            "Model": "M2 (Interacciones Tec21)",
            "AUC-ROC": m2_metrics["roc_auc"],
            "PR-AUC": m2_metrics["pr_auc"],
            "Precision": m2_metrics["precision"],
            "Recall": m2_metrics["recall"],
            "F1": m2_metrics["f1"],
            "Interpretabilidad": "Media-Alta",
            "Tiempo entrenamiento (s)": m2_train_time,
            "Ventajas": (
                "Identifica variables cuyo peso cambia bajo Tec21"
            ),
            "Limitaciones": (
                "Ganancia predictiva marginal; mayor complejidad que M1"
            ),
        },
        {
            "Model": "M3 (Multinivel Jerárquico)",
            "AUC-ROC": m3_metrics["roc_auc"],
            "PR-AUC": m3_metrics["pr_auc"],
            "Precision": m3_metrics["precision"],
            "Recall": m3_metrics["recall"],
            "F1": m3_metrics["f1"],
            "Interpretabilidad": "Media",
            "Tiempo entrenamiento (s)": m3_train_time,
            "Ventajas": (
                "Modela heterogeneidad estructural por escuela"
            ),
            "Limitaciones": (
                "Más costoso; inferencia variacional aproximada"
            ),
        },
    ]
)

comparison_table = comparison_table.sort_values(
    "PR-AUC",
    ascending=False,
).reset_index(drop=True)

display(comparison_table)
comparison_table.to_csv(
    ARTIFACTS / "comparison_table_validation.csv",
    index=False,
)

## Curvas y Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Auditoría de Desempeño: M1 vs M2 vs M3 (Validación)", fontsize=16, fontweight="bold")

# Diccionarios de datos para iterar limpiamente
models_data = [
    {"name": "M1 (Baseline)", "probs": prob_val_m1, "thr": thr_m1, "metrics": m1_metrics},
    {"name": "M2 (Interacciones)", "probs": prob_val_m2, "thr": thr_m2, "metrics": m2_metrics},
    {"name": "M3 (Multinivel)", "probs": prob_val_m3, "thr": thr_m3, "metrics": m3_metrics}
]

for i, mod in enumerate(models_data):
    # 1. Matrices de Confusión (Fila 1)
    cm_display = ConfusionMatrixDisplay(confusion_matrix=mod["metrics"]["cm"], display_labels=["Retenido", "Desertor"])
    cm_display.plot(ax=axes[0, i], cmap="Blues", colorbar=False)
    axes[0, i].set_title(f"{mod['name']}\nUmbral: {mod['thr']:.3f} (Recall: {mod['metrics']['recall']:.2%})", fontsize=11)
    
    # 2. Curvas ROC (Fila 2, compartida)
    RocCurveDisplay.from_predictions(y_val, mod["probs"], ax=axes[1, 0], name=mod["name"])
    
    # 3. Curvas Precision-Recall (Fila 2, compartida)
    PrecisionRecallDisplay.from_predictions(y_val, mod["probs"], ax=axes[1, 1], name=mod["name"])

# Configuración de estilización de curvas
axes[1, 0].plot([0, 1], [0, 1], "k--", label="Azar (AUC = 0.50)")
axes[1, 0].set_title("Curvas ROC", fontsize=12, fontweight="bold")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Línea base para PR (proporción de positivos)
tasa_base = y_val.mean()
axes[1, 1].axhline(y=tasa_base, color="r", linestyle="--", label=f"Azar (Base: {tasa_base:.2%})")
axes[1, 1].set_title("Curvas Precision-Recall", fontsize=12, fontweight="bold")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Ocultar el último eje sobrante (limpieza visual)
axes[1, 2].axis("off")

plt.tight_layout()
plt.savefig(ARTIFACTS / "auditoria_grafica_modelos.png", dpi=300)
plt.show()

## Curvas de aprendizaje

In [ ]:
plot_learning_curve(
    estimator=m1_best,
    X=X_train,
    y=y_train,
    cv=cv,
    title="Curva de aprendizaje - M1 (Baseline)",
    save_path=ARTIFACTS / "m1_learning_curve.png",
)

plot_learning_curve(
    estimator=m2_best,
    X=X_train_m2,
    y=y_train,
    cv=cv,
    title="Curva de aprendizaje - M2 (Interacciones Tec21)",
    save_path=ARTIFACTS / "m2_learning_curve.png",
)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(approx.hist, color="teal", alpha=0.7)
plt.title("Diagnóstico de Convergencia M3: Evolución del ELBO", fontsize=12, fontweight="bold")
plt.xlabel("Iteraciones de Optimización")
plt.ylabel("Loss (Neg ELBO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS / "m3_advi_convergence.png", dpi=300)
plt.show()

## Coeficientes M2

In [ ]:
# Guardar los coeficientes del M2 (Régimen Explícito) para análisis de interpretabilidad en la fase de Robustez
m2_coefs = pd.DataFrame({
    "feature": preprocessor_m2.get_feature_names_out(),
    "coef": m2_best.named_steps["clf"].coef_[0]
}).sort_values("coef", key=abs, ascending=False)
m2_coefs.to_csv(ARTIFACTS / "m2_coefficients.csv", index=False)
display(m2_coefs)

## Coeficientes M3

In [ ]:
m3_beta_summary = az.summary(trace_m3, var_names=["beta"])

display(m3_beta_summary.info())

In [ ]:
m3_beta_summary = az.summary(trace_m3, var_names=["beta"])
m3_beta_summary["feature"] = feature_names_m3

# Usamos las columnas ETI al 89% que genera tu entorno
m3_beta_summary = m3_beta_summary[["feature", "mean", "sd", "eti89_lb", "eti89_ub"]]

# Convertimos de string a float para permitir análisis matemático posterior
cols_to_float = ["mean", "sd", "eti89_lb", "eti89_ub"]
m3_beta_summary[cols_to_float] = m3_beta_summary[cols_to_float].astype(float)

m3_beta_summary.to_csv(ARTIFACTS / "m3_fixed_effects_beta.csv", index=False)


# 2. Extraer la Heterogeneidad Estructural (Efectos por Escuela)
m3_alpha_summary = az.summary(trace_m3, var_names=["alpha"])
m3_delta_summary = az.summary(trace_m3, var_names=["delta"])

# Recuperar los nombres de las escuelas basados en tu índice categórico
schools_names = sorted(df[SCHOOL_COL].unique())

m3_school_effects = pd.DataFrame({
    "school": schools_names,
    
    # Riesgo base por escuela (Pre-Tec21)
    "alpha_mean": m3_alpha_summary["mean"].astype(float).values, 
    "alpha_eti_low": m3_alpha_summary["eti89_lb"].astype(float).values,
    "alpha_eti_high": m3_alpha_summary["eti89_ub"].astype(float).values,
    
    # Impacto estructural diferencial del Tec21
    "delta_mean": m3_delta_summary["mean"].astype(float).values, 
    "delta_eti_low": m3_delta_summary["eti89_lb"].astype(float).values,
    "delta_eti_high": m3_delta_summary["eti89_ub"].astype(float).values,
})

# Ordenar por el impacto del Tec21 para ver qué escuelas cambiaron más
m3_school_effects = m3_school_effects.sort_values("delta_mean")
m3_school_effects.to_csv(ARTIFACTS / "m3_random_effects_schools.csv", index=False)

print("── Top 3 Escuelas donde Tec21 redujo más la deserción estructural ──")
display(m3_school_effects.head(3))

print("\n── Top 3 Escuelas con mayor resistencia estructural al Tec21 ──")
display(m3_school_effects.tail(3))

## Análisis de Errores

In [ ]:
ERROR_NUM_COLS = [
    "PNA",
    "scholarship.perc",
    "activity_count_unified",
    "admission.test",
]

audit_m1, audit_m1_era, audit_m1_school = error_audit(
    X_val,
    y_val,
    prob_val_m1,
    thr_m1,
    school_col=SCHOOL_COL,
    era_col=ERA_COL,
    numeric_cols=ERROR_NUM_COLS,
)

audit_m2, audit_m2_era, audit_m2_school = error_audit(
    X_val,
    y_val,
    prob_val_m2,
    thr_m2,
    school_col=SCHOOL_COL,
    era_col=ERA_COL,
    numeric_cols=ERROR_NUM_COLS,
)

audit_m3, audit_m3_era, audit_m3_school = error_audit(
    X_val,
    y_val,
    prob_val_m3,
    thr_m3,
    school_col=SCHOOL_COL,
    era_col=ERA_COL,
    numeric_cols=ERROR_NUM_COLS,
)

print("Distribución de errores por era - M1")
display(audit_m1_era)

print("Distribución de errores por era - M2")
display(audit_m2_era)

print("Distribución de errores por era - M3")
display(audit_m3_era)

print("Top 10 escuelas con mayor FN rate - M1")
display(audit_m1_school.head(10))

print("Top 10 escuelas con mayor FN rate - M2")
display(audit_m2_school.head(10))

print("Top 10 escuelas con mayor FN rate - M3")
display(audit_m3_school.head(10))

In [ ]:
error_summary = pd.DataFrame(
    [
        audit_m1["error_type"].value_counts().rename("M1"),
        audit_m2["error_type"].value_counts().rename("M2"),
        audit_m3["error_type"].value_counts().rename("M3"),
    ]
).fillna(0).astype(int)

display(error_summary)
error_summary.to_csv(ARTIFACTS / "error_summary_validation.csv")

# Exportar modelos 

In [ ]:
print("Exportando modelos, preprocesadores y matrices de prueba...")

# Extraer los preprocesadores que SÍ fueron ajustados (viven dentro de los pipelines)
fitted_prep_m1 = m1_best.named_steps["prep"]
fitted_prep_m2 = m2_best.named_steps["prep"]

# Exportar Modelos Scikit-Learn y Preprocesadores
joblib.dump(m1_best, ARTIFACTS / "m1_lr_baseline.pkl")
joblib.dump(m2_best, ARTIFACTS / "m2a_lr_era.pkl")  # m2_best actual mapea al m2a_lr_era

joblib.dump(preprocessor_m1, ARTIFACTS / "preprocessor_common.pkl")
joblib.dump(preprocessor_m2, ARTIFACTS / "preprocessor_m3.pkl")  # Usando preprocessor_m2 para M3

# Extraer medias del M3 (Asumiendo que 'trace_m3' es tu objeto InferenceData)
alpha_school_mean = trace_m3.posterior["alpha"].mean(dim=["chain", "draw"]).values
beta_mean = trace_m3.posterior["beta"].mean(dim=["chain", "draw"]).values

# Dependiendo de la especificación final de tu M3:
if "delta" in trace_m3.posterior:
    # Si usaste pendiente aleatoria por escuela
    beta_era_mean = trace_m3.posterior["delta"].mean(dim=["chain", "draw"]).values 
else:
    # Si usaste un efecto fijo global para la era
    beta_era_mean = float(trace_m3.posterior["beta_era"].mean(dim=["chain", "draw"]))

np.savez(
    ARTIFACTS / "m3_multilevel_advi.npz", 
    alpha_school=alpha_school_mean, 
    beta_era=beta_era_mean, 
    beta=beta_mean
)

# Transformar los datos de Test usando los preprocesadores ajustados
X_test_sk = fitted_prep_m1.transform(X_test).astype("float32")
X_test_m3 = fitted_prep_m2.transform(add_m2_features(X_test)).astype("float32")

# Transformar una muestra de Train para el background de SHAP (evita explosión de memoria)
X_train_sk = fitted_prep_m1.transform(X_train).astype("float32")
X_train_sk_sample = X_train_sk[:2000]

# Extraer arreglos estructurales
era_test = X_test[ERA_COL].values.astype("int32")
school_test = X_test[SCHOOL_COL].map(school_index).values.astype("int32")
era_train = X_train[ERA_COL].values.astype("int32")

np.savez(
    ARTIFACTS / "test_arrays.npz",
    X_test_sk=X_test_sk,
    X_test_m3=X_test_m3,
    y_test=y_test,
    era_test=era_test,
    school_test=school_test,
    X_train_sk_sample=X_train_sk_sample,
    y_train=y_train,
    era_train=era_train
)

# Exportar Nombres de Variables y Etiquetas
feature_names = {
    "common": list(fitted_prep_m1.get_feature_names_out()),
    "m3": list(fitted_prep_m2.get_feature_names_out()),
    "era_labels": ["Pre-Tec21", "Tec21"],
    "school_labels": sorted(df["school"].unique())  # Mantiene el orden alfabético original
}

with open(ARTIFACTS / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, indent=4, ensure_ascii=False)

print("¡Artefactos guardados exitosamente!")

In [ ]:
model_selection_summary = pd.DataFrame(
    [
        {
            "Model": "M1",
            "Best CV PR-AUC": m1_search.best_score_,
            "Best CV std": pd.DataFrame(m1_search.cv_results_).loc[
                m1_search.best_index_,
                "std_test_score",
            ],
            "Best params": json.dumps(m1_search.best_params_, default=str),
        },
        {
            "Model": "M2",
            "Best CV PR-AUC": m2_search.best_score_,
            "Best CV std": pd.DataFrame(m2_search.cv_results_).loc[
                m2_search.best_index_,
                "std_test_score",
            ],
            "Best params": json.dumps(m2_search.best_params_, default=str),
        },
        {
            "Model": "M3",
            "Best CV PR-AUC": np.nan,
            "Best CV std": np.nan,
            "Best params": json.dumps(
                {
                    "inference": "ADVI",
                    "advi_steps": 30000,
                    "posterior_samples": 1000,
                    "random_effects": [
                        "alpha_school",
                        "delta_school_x_era",
                    ],
                }
            ),
        },
    ]
)

display(model_selection_summary)
model_selection_summary.to_csv(
    ARTIFACTS / "model_selection_summary.csv",
    index=False,
)